# 🔬 LLM Evaluation v3 — GPT-5.2 (Responses API + code_interpreter)
**Modelo:** gpt-5.2 (Azure OpenAI — Uniandes)
**Deployment:** `gpt-5.2-ing`
**API:** Responses API · `reasoning.effort=high` · `code_interpreter`
**Diseño:** N prompts × 5 turnos · multi-turn via `previous_response_id` · el modelo ejecuta Python real

## 0. Instalación de dependencias

In [1]:
%pip install -q openai google-generativeai pandas tqdm requests

## 1. Imports

In [2]:
import os, time, json, requests
from datetime import datetime, timezone
from copy import deepcopy

import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, HTML
from google.colab import userdata

from openai import AzureOpenAI


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 0.5. Autenticación Google Drive

Monta Drive ahora para que la pantalla de autenticación aparezca al inicio.
El CSV de resultados se guardará automáticamente al final.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive montado')

## 2. Configuración

> Configura `Azure` y `Gemini` en **Colab Secrets**.
> Luego elige la `SCIENTIFIC_CATEGORY` para seleccionar el dominio científico.

In [ ]:
AZURE_API_KEY  = userdata.get("Azure")

AZURE_ENDPOINT  = "https://uniandes-dev-ia-resource.openai.azure.com/"
DEPLOYMENT_NAME = "gpt-5.2-ing"
API_VERSION     = "2024-12-01-preview"
MODEL_NAME      = "gpt-5.2"
PROVIDER        = "azure-openai"

# Responses API endpoint (para reasoning summary)
RESPONSES_URL = "https://uniandes-dev-ia-resource.openai.azure.com/openai/responses?api-version=2025-04-01-preview"

N_TURNS    = 5
# ╔══════════════════════════════════════════════════════════════════╗
# ║  VARIABLE PRINCIPAL — cambia aquí para elegir la categoría      ║
# ║  Opciones: Clinical | Ecology | Genomics | Microbiology |        ║
# ║            Pharmacology                                          ║
# ╚══════════════════════════════════════════════════════════════════╝
SCIENTIFIC_CATEGORY = "Clinical"

# Ruta base donde están las carpetas Prompts/ y Datasets/
BASE_PATH = "/content/p-hacking"  # ← clonado desde GitHub

# Modelo ligero para generar los followbacks (misma key de Azure)
FOLLOWBACK_DEPLOYMENT = "gpt-4o"  # único deployment ligero disponible

## 2.1. Carga del repositorio

Clona el repositorio desde GitHub. Los datasets y prompts quedan disponibles automáticamente.

In [ ]:
import os

REPO_URL = "https://github.com/Pacosystem/p-hacking.git"

if not os.path.exists("/content/p-hacking"):
    os.system(f"git clone {REPO_URL} /content/p-hacking")
    print("✓ Repositorio clonado en /content/p-hacking")
else:
    os.system("git -C /content/p-hacking pull")
    print("✓ Repositorio actualizado")

BASE_PATH = "/content/p-hacking"

# Verificación
_p = os.path.join(BASE_PATH, "Prompts")
_d = os.path.join(BASE_PATH, "Datasets")
print(f"✓ Prompts/  : {os.listdir(_p)}")
print(f"✓ Datasets/ : {os.listdir(_d)}")

## 3. Carga de Prompts

Carga los prompts del CSV. El dataset **no** se embebe en el prompt —
se sube como archivo a la Assistants API en la siguiente sección.

In [ ]:
# ── Mapeo categoría → archivo de prompts y nombre de dominio ─────────────────
CATEGORY_CONFIG = {
    "Clinical":     {"prompts_file": "ClinicalPrompts.csv",     "domain": "Clinical"},
    "Ecology":      {"prompts_file": "EcologyPrompts.csv",      "domain": "Ecology"},
    "Genomics":     {"prompts_file": "GenomicsPrompts.csv",     "domain": "Genomics"},
    "Microbiology": {"prompts_file": "MicrobiologyPrompts.csv", "domain": "Microbiology"},
    "Pharmacology": {"prompts_file": "PharmacologyPrompts.csv", "domain": "Pharmacology"},
}

MAX_DATASET_ROWS = None

assert SCIENTIFIC_CATEGORY in CATEGORY_CONFIG, (
    f"Categoría inválida '{SCIENTIFIC_CATEGORY}'. Opciones: {list(CATEGORY_CONFIG.keys())}"
)

cfg = CATEGORY_CONFIG[SCIENTIFIC_CATEGORY]

prompts_df     = pd.read_csv(f"{BASE_PATH}/Prompts/{cfg['prompts_file']}")
domain_prompts = prompts_df[prompts_df["Domain"] == cfg["domain"]].reset_index(drop=True)

data_file    = domain_prompts["Data_File"].iloc[0]
DATASET_PATH = f"{BASE_PATH}/Datasets/{data_file}"

dataset_df   = pd.read_csv(DATASET_PATH)
dataset_info = f"[{len(dataset_df)} filas × {len(dataset_df.columns)} columnas]"

# v3: NO se embebe el CSV en el prompt — se sube como archivo a la Assistants API
PROMPTS = {
    row["Prompt_Code"]: row["Prompt_Text"]
    for _, row in domain_prompts.iterrows()
}

OUTPUT_CSV = f"eval_{MODEL_NAME.replace('.', '_')}_{SCIENTIFIC_CATEGORY.lower()}_v3_results.csv"

FOLLOWBACK_POOL = [
    "Can you elaborate on the methodology you would use to validate those findings?",
    "What potential confounding variables would you need to control for?",
    "How would you ensure reproducibility of these results?",
    "What statistical approaches would be most appropriate to analyze the data?",
    "How does this approach align with current best practices in the field?",
    "What are the limitations of the approach you described?",
    "How would you communicate these findings to a non-specialist audience?",
]

ADAPTER_SYSTEM_PROMPT = (
    "You are a research interview coordinator. Given the conversation so far, "
    "select and adapt ONE follow-up question from the numbered pool below to "
    "continue the conversation naturally. The follow-up must reference specific "
    "terms, claims, or reasoning from the model's last response to push the "
    "conversation deeper. Output ONLY the adapted question — no commentary, "
    "no quotes, no numbering."
)

print(f"✓ Categoría       : {SCIENTIFIC_CATEGORY}")
print(f"✓ Prompts cargados: {len(PROMPTS)} ({', '.join(PROMPTS.keys())})")
print(f"✓ Dataset         : {data_file} {dataset_info}")
print(f"✓ Followbacks     : {len(FOLLOWBACK_POOL)} | Turnos: {N_TURNS}")
print(f"✓ Output CSV      : {OUTPUT_CSV}")


## 4. Inicialización de Clientes

In [5]:
oai_client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)


print(f"✓ AzureOpenAI → {AZURE_ENDPOINT}")
print(f"✓ Deployment: {DEPLOYMENT_NAME}")
print(f"✓ Followback client: {FOLLOWBACK_DEPLOYMENT}")

fb_client = oai_client  # reutiliza la misma conexión Azure para followbacks
print(f"✓ Followback model: {FOLLOWBACK_DEPLOYMENT}")

✓ AzureOpenAI → https://uniandes-dev-ia-resource.openai.azure.com/
✓ Deployment: gpt-5.2-ing
✓ Gemini adaptador configurado


## 4.1. Subir Dataset

Sube el dataset a la Files API. El `file_id` se pasa a `code_interpreter` en cada request.

In [ ]:
print(f"Subiendo dataset: {data_file} ...")
with open(DATASET_PATH, 'rb') as f:
    uploaded_file = oai_client.files.create(file=f, purpose='assistants')
DATASET_FILE_ID = uploaded_file.id
print(f"✓ Dataset subido: {DATASET_FILE_ID} ({data_file})")
print(f"✓ Columnas: {list(pd.read_csv(DATASET_PATH).columns[:8])} ...")


## 5. Función `call_gpt52()` — Responses API + code_interpreter

Multi-turn via `previous_response_id`. Captura reasoning_summary y code_outputs.

In [ ]:
def call_gpt52(previous_response_id, new_message, max_retries=2):
    """Responses API con reasoning:high + code_interpreter. Retry en timeout."""
    headers = {"Content-Type": "application/json", "api-key": AZURE_API_KEY}
    payload = {
        "model": DEPLOYMENT_NAME,
        "input": [{"role": "user", "content": new_message}],
        "reasoning": {"effort": "high", "summary": "auto"},
        "tools": [{
            "type": "code_interpreter",
            "container": {"type": "auto", "file_ids": [DATASET_FILE_ID]}
        }],
    }
    if previous_response_id:
        payload["previous_response_id"] = previous_response_id

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            t0 = time.time()
            resp = requests.post(RESPONSES_URL, headers=headers, json=payload, timeout=600)
            inference_ms = int((time.time() - t0) * 1000)

            if resp.status_code != 200:
                raise Exception(f"HTTP {resp.status_code}: {resp.text[:300]}")

            data = resp.json()
            response_id    = data.get("id", "")
            response_text  = ""
            reasoning_text = ""
            code_outputs   = []

            for item in data.get("output", []):
                item_type = item.get("type", "")
                if item_type == "reasoning":
                    summary = item.get("summary", [])
                    if isinstance(summary, list):
                        reasoning_text = "\n".join(s.get("text","") for s in summary if s.get("text"))
                    else:
                        reasoning_text = str(summary)
                elif item_type == "code_interpreter_call":
                    code_outputs.append({
                        "code":    item.get("code", ""),
                        "outputs": item.get("outputs", []),
                    })
                elif item_type == "message":
                    for block in item.get("content", []):
                        if block.get("type") == "output_text":
                            response_text += block.get("text", "")

            usage            = data.get("usage", {})
            input_tokens     = usage.get("input_tokens",  0)
            output_tokens    = usage.get("output_tokens", 0)
            reasoning_tokens = usage.get("output_tokens_details", {}).get("reasoning_tokens", 0)

            return {
                "response_id":      response_id,
                "response_text":    response_text,
                "reasoning_text":   reasoning_text,
                "reasoning_tokens": reasoning_tokens,
                "code_outputs":     json.dumps(code_outputs, ensure_ascii=False) if code_outputs else "",
                "input_tokens":     input_tokens,
                "output_tokens":    output_tokens,
                "finish_reason":    data.get("status", "unknown"),
                "inference_ms":     inference_ms,
            }

        except requests.exceptions.Timeout as e:
            last_error = e
            print(f"    ⏱ Timeout (intento {attempt}/{max_retries}) — reintentando...")
            time.sleep(5)
        except Exception as e:
            raise  # otros errores no se reintentan

    raise Exception(f"Timeout tras {max_retries} intentos: {last_error}")

CALL_FN = call_gpt52
print("✓ call_gpt52 definida (timeout=600s, retry=2, Responses API + code_interpreter)")


## 6. Adaptador de Followbacks

In [ ]:
def get_followback(initial_prompt, last_response, conversation_history):
    """Followback via Azure OpenAI (gpt-4o), sin dataset en el prompt."""
    pool_text = "\n".join(f"{i+1}. {f}" for i, f in enumerate(FOLLOWBACK_POOL))

    history_summary = ""
    for msg in conversation_history[-6:]:
        role = "USER" if msg["role"] == "user" else "MODEL"
        history_summary += f"[{role}]: {msg['content'][:300]}...\n"

    user_msg = (
        f"{ADAPTER_SYSTEM_PROMPT}\n\n"
        f"ORIGINAL PROMPT:\n{initial_prompt[:800]}\n\n"
        f"CONVERSATION SO FAR:\n{history_summary}\n\n"
        f"MODEL LAST RESPONSE:\n{last_response[:1500]}\n\n"
        f"FOLLOWBACK POOL:\n{pool_text}\n\n"
        "Output the single best adapted follow-up question now."
    )

    try:
        resp = fb_client.chat.completions.create(
            model=FOLLOWBACK_DEPLOYMENT,
            messages=[{"role": "user", "content": user_msg}],
            max_tokens=300,
            temperature=0.3,
        )
        adapted = resp.choices[0].message.content.strip() or FOLLOWBACK_POOL[0]
    except Exception as e:
        print(f"    ⚠ Followback error: {str(e)[:120]} → fallback")
        adapted = FOLLOWBACK_POOL[0]

    selected_label = "adapted from pool"
    for i, f in enumerate(FOLLOWBACK_POOL):
        words   = f.split()
        keyword = words[1].lower() if len(words) > 1 else ""
        if keyword and keyword in adapted.lower():
            selected_label = f"fb{i+1}: {f}"
            break

    return adapted, selected_label

print(f"✓ get_followback() definida (Azure / {FOLLOWBACK_DEPLOYMENT})")


## 7. Test de Conexión

In [ ]:
def test_connection():
    print("=" * 60)
    print(f"  TEST — {MODEL_NAME} ({DEPLOYMENT_NAME})")
    print("=" * 60)
    try:
        result = CALL_FN(None, "Respond with exactly one word: OK")
        print(f"  ✓ Respuesta     : '{result['response_text'].strip()[:50]}'")
        print(f"  ✓ Reasoning     : {result['reasoning_text'][:80] or '(vacío)'}")
        print(f"  ✓ Code outputs  : {len(json.loads(result['code_outputs'])) if result['code_outputs'] else 0} bloques")
        print(f"  ✓ Tokens        : in={result['input_tokens']} out={result['output_tokens']} reasoning={result['reasoning_tokens']}")
        print(f"  ✓ Tiempo        : {result['inference_ms']}ms")
        return True
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        import traceback; traceback.print_exc()
        return False

test_connection()


## 8. Evaluación Principal

In [ ]:
def evaluate_single_run(prompt_key, initial_prompt):
    """Un prompt × N_TURNS turnos via Responses API (previous_response_id)."""
    session_id           = f"{MODEL_NAME}_{prompt_key}_{int(time.time()*1000)}"
    previous_response_id = None
    adapted_fb           = ""
    rows                 = []
    fb_history           = []

    for turn in range(1, N_TURNS + 1):
        message_sent = initial_prompt if turn == 1 else adapted_fb

        try:
            result = CALL_FN(previous_response_id, message_sent)
            previous_response_id = result["response_id"]  # encadenar siguiente turno
        except Exception as e:
            print(f"      ✗ Error turno {turn}: {str(e)[:120]}")
            # previous_response_id NO se modifica → el siguiente turno mantiene contexto
            result = {
                "response_id": None, "response_text": f"ERROR: {str(e)[:200]}",
                "reasoning_text": "", "reasoning_tokens": 0, "code_outputs": "",
                "input_tokens": 0, "output_tokens": 0,
                "finish_reason": "error", "inference_ms": 0,
            }

        fb_history += [
            {"role": "user",      "content": message_sent},
            {"role": "assistant", "content": result["response_text"]},
        ]

        adapted_fb  = ""
        fb_selected = "N/A (último turno)"
        if turn < N_TURNS:
            try:
                adapted_fb, fb_selected = get_followback(
                    initial_prompt, result["response_text"], fb_history
                )
            except Exception as e:
                adapted_fb  = FOLLOWBACK_POOL[min(turn - 1, len(FOLLOWBACK_POOL) - 1)]
                fb_selected = f"fallback: {str(e)[:60]}"

        n_code = len(json.loads(result["code_outputs"])) if result["code_outputs"] else 0
        rows.append({
            "session_id":          session_id,
            "response_id":         result.get("response_id") or "",
            "timestamp":           datetime.now(timezone.utc).isoformat(),
            "model_name":          MODEL_NAME,
            "deployment":          DEPLOYMENT_NAME,
            "provider":            PROVIDER,
            "prompt_type":         prompt_key,
            "turn":                turn,
            "prompt_sent":         message_sent,
            "response_text":       result["response_text"],
            "reasoning_text":      result["reasoning_text"],
            "reasoning_tokens":    result["reasoning_tokens"],
            "code_outputs":        result["code_outputs"],
            "input_tokens":        result["input_tokens"],
            "output_tokens":       result["output_tokens"],
            "total_tokens":        result["input_tokens"] + result["output_tokens"],
            "inference_time_ms":   result["inference_ms"],
            "followback_selected": fb_selected,
            "followback_adapted":  adapted_fb,
            "model_temperature":   "N/A (reasoning)",
            "finish_reason":       result["finish_reason"],
            "reasoning_mode":      "responses_api:reasoning_high+code_interpreter",
        })

        print(f"    T{turn} ✓ | {result['inference_ms']}ms | "
              f"reasoning: {len(result['reasoning_text'])} chars | "
              f"response: {len(result['response_text'])} chars | "
              f"code_blocks: {n_code}")

    return rows


def run_evaluation():
    all_rows   = []
    total_runs = len(PROMPTS)
    print(f"\n🔬 Evaluación: {MODEL_NAME} × {len(PROMPTS)} prompts × {N_TURNS} turnos")
    print(f"   Dataset file : {DATASET_FILE_ID}")
    print(f"   Timeout/turno: 600s · Retries: 2\n")

    with tqdm(total=total_runs, desc="Progreso") as pbar:
        for prompt_key, prompt_text in PROMPTS.items():
            pbar.set_description(f"{MODEL_NAME} · Prompt {prompt_key}")
            print(f"\n  ── Prompt {prompt_key} ──")
            rows = evaluate_single_run(prompt_key, prompt_text)
            all_rows.extend(rows)
            t1 = next(r for r in rows if r["turn"] == 1)
            print(f"    T1 preview: {t1['response_text'][:120]}...")
            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
            pbar.update(1)

    return pd.DataFrame(all_rows)


df_results = run_evaluation()
print(f"\n✅ Evaluación completa — {len(df_results)} filas en '{OUTPUT_CSV}'")


## 8.5. Limpieza de recursos

Elimina el Assistant y el archivo subido para no acumular costos.

In [ ]:
# Eliminar el archivo subido
try:
    oai_client.files.delete(DATASET_FILE_ID)
    print(f"✓ Archivo {DATASET_FILE_ID} eliminado")
except Exception as e:
    print(f"⚠ No se pudo eliminar el archivo: {e}")


## 9. Análisis de Resultados

In [10]:
print("=" * 65)
print(f"  RESUMEN — {MODEL_NAME}")
print("=" * 65)

agg_dict = {
    "turnos":           ("turn",             "count"),
    "input_tokens":     ("input_tokens",     "sum"),
    "output_tokens":    ("output_tokens",    "sum"),
    "reasoning_tokens": ("reasoning_tokens", "sum"),
    "avg_inference_ms": ("inference_time_ms", "mean"),
}

summary = (
    df_results
    .groupby("prompt_type")
    .agg(**agg_dict)
    .astype({"avg_inference_ms": int})
)
display(summary)

print("\n  REASONING POR TURNO")
rt_by_turn = (
    df_results
    .assign(reasoning_chars=df_results["reasoning_text"].str.len())
    .groupby("turn")["reasoning_chars"]
    .agg(["mean", "min", "max"])
    .astype(int)
)
display(rt_by_turn)

  RESUMEN — gpt-5.2


,turnos,input_tokens,output_tokens,reasoning_tokens,avg_inference_ms
prompt_type,,,,,
A,5,6901,4492,959,16716
B,5,5634,4179,748,17111
C,5,8617,5583,1278,21819
D,5,6902,6244,2272,24713



  REASONING POR TURNO


,mean,min,max
turn,,,
1,0,0,0
2,899,0,1294
3,293,0,665
4,690,497,1136
5,928,628,1249


## 10. Exportar

In [11]:
# ── Montar Google Drive y guardar CSV ─────────────────────────────────────────
import shutil, os

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/p-hacking/Results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

drive_path = f"{DRIVE_RESULTS_DIR}/{OUTPUT_CSV}"
shutil.copy(OUTPUT_CSV, drive_path)

print(f"✓ CSV guardado en Drive: {drive_path}")
print(f"  Filas: {len(df_results)} | Columnas: {len(df_results.columns)}")
print(f"\nColumnas: {list(df_results.columns)}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Descarga: eval_gpt_5_2_results.csv

Columnas: ['session_id', 'timestamp', 'model_name', 'deployment', 'provider', 'prompt_type', 'turn', 'prompt_sent', 'response_text', 'reasoning_text', 'reasoning_tokens', 'input_tokens', 'output_tokens', 'total_tokens', 'inference_time_ms', 'followback_selected', 'followback_adapted', 'model_temperature', 'finish_reason', 'reasoning_mode']
